# 08 — Backward Factor Trace on a Tiny Vision Transformer

This notebook tests whether the **Backward Factor Trace** algorithm (introduced in NB02 for MLPs, extended to CNNs in NB07) transfers to transformer architectures.

**Setup:** A tiny ViT is trained on MNIST even/odd (same binary task as NB02). We then run the backward trace through the FFN sublayers of each transformer block, using the **CLS token activation** as the per-sample feature vector at each layer boundary.

**Key adaptation:** The algorithm works on any `(W, act_input)` pair. Transformer FFN sublayers are standard linear layers — no modification needed. The only choice is which token to trace (CLS) and which layers to include (FFN only; attention is left for future work).

## 1. Imports & Setup

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib

from src import bft
from src.data_utils import get_mnist_loaders
from src.training import label_transform_even_odd
from src.plot_utils import plot_scaffold_graph

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

_nb_dir   = os.path.dirname(os.path.abspath('__file__'))
_repo_dir = os.path.dirname(_nb_dir)
FIG_DIR   = os.path.join(_repo_dir, 'figs', '05_transformer_trace')
os.makedirs(FIG_DIR, exist_ok=True)

matplotlib.rcParams.update({'figure.dpi': 80})
print(f'FIG_DIR: {FIG_DIR}')


## 2. Dataset

In [ ]:
train_loader, test_loader = get_mnist_loaders(batch_size=256, root='../data/')
CLASS_NAMES = {0: 'even', 1: 'odd'}
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

## 3. Tiny ViT Architecture

```
28×28 image → 16 patches of 7×7 → embed (49→64) + CLS + pos_embed
TransformerBlock × 2:
  LN → MultiHeadAttention(dim=64, heads=2) → residual
  LN → FFN: Linear(64→128) → GELU → Linear(128→64) → residual
LN → CLS token → Linear(64, 2) → log_softmax
```

Total ~100k parameters. Each `TransformerBlock` stores FFN intermediates when `capture=True`.

In [ ]:
class PatchEmbedder(nn.Module):
    def __init__(self, img_size=28, patch_size=7, in_channels=1, embed_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2   # 16
        patch_dim = in_channels * patch_size * patch_size  # 49
        self.proj = nn.Linear(patch_dim, embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size
        x = x.unfold(2, p, p).unfold(3, p, p)         # (B, C, H/p, W/p, p, p)
        x = x.contiguous().view(B, C, -1, p * p)       # (B, C, n_patches, p*p)
        x = x.permute(0, 2, 1, 3).contiguous().view(B, -1, C * p * p)  # (B, n_patches, C*p*p)
        return self.proj(x)                             # (B, n_patches, embed_dim)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=64, n_heads=2, ffn_dim=128, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, n_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn1 = nn.Linear(embed_dim, ffn_dim)
        self.ffn2 = nn.Linear(ffn_dim, embed_dim)
        self.drop = nn.Dropout(dropout)
        self._capture = False
        self._ffn1_in = None
        self._ffn1_out = None
        self._ffn2_in = None
        self._ffn2_out = None
        self._attn_weights = None
        self._attn_in_tokens = None  # (B, T, d) all tokens entering MHA, post-LN1

    def forward(self, x):
        h = self.ln1(x)
        if self._capture:
            self._attn_in_tokens = h.detach()  # store before MHA mixes tokens
        attn_out, attn_w = self.attn(h, h, h,
                                     need_weights=self._capture,
                                     average_attn_weights=False)
        if self._capture:
            self._attn_weights = attn_w.detach()   # (B, heads, T, T)
        x = x + attn_out
        h = self.ln2(x)
        if self._capture:
            self._ffn1_in = h.detach()
        h2 = F.gelu(self.ffn1(h))
        if self._capture:
            self._ffn1_out = h2.detach()
            self._ffn2_in = h2.detach()
        h3 = self.ffn2(h2)
        if self._capture:
            self._ffn2_out = h3.detach()
        x = x + self.drop(h3)
        return x


class TinyViT(nn.Module):
    def __init__(self, img_size=28, patch_size=7, in_channels=1,
                 embed_dim=64, n_heads=2, ffn_dim=128, n_blocks=2, n_classes=2):
        super().__init__()
        self.patch_embed = PatchEmbedder(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, n_heads, ffn_dim) for _ in range(n_blocks)
        ])
        self.ln = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, n_classes)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x, capture=False):
        for blk in self.blocks:
            blk._capture = capture
        B = x.shape[0]
        tokens = torch.cat([self.cls_token.expand(B, -1, -1),
                             self.patch_embed(x)], dim=1) + self.pos_embed
        for blk in self.blocks:
            tokens = blk(tokens)
        return F.log_softmax(self.head(self.ln(tokens[:, 0])), dim=1)


model = TinyViT().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.NLLLoss()
N_EPOCHS = 12
train_losses, test_accs = [], []

for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0.0
    for imgs, digits in train_loader:
        imgs = imgs.to(device)
        labels = label_transform_even_odd(digits).to(device)
        loss = criterion(model(imgs), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    n_correct = 0
    with torch.no_grad():
        for imgs, digits in test_loader:
            imgs = imgs.to(device)
            labels = label_transform_even_odd(digits).to(device)
            n_correct += (model(imgs).argmax(1) == labels).sum().item()
    acc = n_correct / len(test_loader.dataset)
    train_losses.append(total_loss / len(train_loader))
    test_accs.append(acc)
    print(f'Epoch {epoch+1:2d} | loss {train_losses[-1]:.4f} | test acc {acc:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 2.8))
ax1.plot(train_losses)
ax1.set(title='Train loss', xlabel='epoch', ylabel='NLL loss')
ax2.plot(test_accs)
ax2.axhline(0.95, ls='--', color='gray', label='95%')
ax2.set(title='Test accuracy', xlabel='epoch', ylim=(0.8, 1.0))
ax2.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'training_curve.png'), dpi=120, bbox_inches='tight')
plt.show()


## 5. Activation Collection

Run the test set through the model with `capture=True`. For each correctly-classified sample,
extract the **CLS token** activation at every FFN layer boundary. This gives `(n_samples, dim)`
arrays — identical in shape to what `trace_single_layer` expects from an MLP.

In [ ]:
model.eval()
all_images, all_targets, all_digits = [], [], []
ffn_acts = {k: [] for k in [
    'b0_f1_in', 'b0_f1_out', 'b0_f2_in', 'b0_f2_out',
    'b1_f1_in', 'b1_f1_out', 'b1_f2_in', 'b1_f2_out',
]}

_capture_map = [
    ('b0_f1_in',  model.blocks[0], '_ffn1_in'),
    ('b0_f1_out', model.blocks[0], '_ffn1_out'),
    ('b0_f2_in',  model.blocks[0], '_ffn2_in'),
    ('b0_f2_out', model.blocks[0], '_ffn2_out'),
    ('b1_f1_in',  model.blocks[1], '_ffn1_in'),
    ('b1_f1_out', model.blocks[1], '_ffn1_out'),
    ('b1_f2_in',  model.blocks[1], '_ffn2_in'),
    ('b1_f2_out', model.blocks[1], '_ffn2_out'),
]

with torch.no_grad():
    for imgs, digits in test_loader:
        imgs = imgs.to(device)
        labels = label_transform_even_odd(digits)
        logits = model(imgs, capture=True)
        mask = (logits.argmax(1).cpu() == labels)
        if not mask.any():
            continue
        for key, blk, attr in _capture_map:
            tensor = getattr(blk, attr)          # (B, T, D)
            ffn_acts[key].append(tensor[mask, 0].cpu().numpy())  # CLS token only
        all_images.append(imgs[mask].cpu().numpy())
        all_targets.append(labels[mask].numpy())
        all_digits.append(digits[mask].numpy())

for k in ffn_acts:
    ffn_acts[k] = np.concatenate(ffn_acts[k], axis=0)
all_images  = np.concatenate(all_images,  axis=0)
all_targets = np.concatenate(all_targets, axis=0)
all_digits  = np.concatenate(all_digits,  axis=0)

print(f'Collected {len(all_targets)} correctly-classified test samples')
for k, v in ffn_acts.items():
    print(f'  {k}: {v.shape}')

## 6. Backward Factor Trace

We trace 4 FFN linear layers in reverse order using `bft` from `src/bft.py`
in layer-dict mode — exactly the same algorithm as for an MLP, but with the
CLS token activations and directly extracted weight matrices.

```
B1-FFN2 (128→64) ← B1-FFN1 (64→128) ← B0-FFN2 (128→64) ← B0-FFN1 (64→128)
```

Stimulus weights start uniform and are propagated backward via the
`factor_project_raw` weighting mode, which projects input activations onto
the arbor-space direction of the top NMF factor.

In [ ]:
# Weight matrices (move to CPU numpy)
W_b0_f1 = model.blocks[0].ffn1.weight.detach().cpu().numpy()  # (128, 64)
W_b0_f2 = model.blocks[0].ffn2.weight.detach().cpu().numpy()  # (64, 128)
W_b1_f1 = model.blocks[1].ffn1.weight.detach().cpu().numpy()  # (128, 64)
W_b1_f2 = model.blocks[1].ffn2.weight.detach().cpu().numpy()  # (64, 128)

LAYER_LABELS = ['B0-FFN1', 'B0-FFN2', 'B1-FFN1', 'B1-FFN2']

layer_dicts = [
    {'type': 'fc', 'name': 'B0-FFN1', 'weight': W_b0_f1, 'input_fmap': ffn_acts['b0_f1_in']},
    {'type': 'fc', 'name': 'B0-FFN2', 'weight': W_b0_f2, 'input_fmap': ffn_acts['b0_f2_in']},
    {'type': 'fc', 'name': 'B1-FFN1', 'weight': W_b1_f1, 'input_fmap': ffn_acts['b1_f1_in']},
    {'type': 'fc', 'name': 'B1-FFN2', 'weight': W_b1_f2, 'input_fmap': ffn_acts['b1_f2_in']},
]

def _spine(root):
    chain, node = [root], root
    while node['children']:
        node = node['children'][0]
        chain.append(node)
    return chain

tree_root = bft(
    layer_dicts, n_branches=1,
    k_max=15, threshold=0.90
)

# _spine returns last-layer-first: [B1-FFN2, B1-FFN1, B0-FFN2, B0-FFN1]
trace_results = _spine(tree_root)

# Add notebook-specific label field for visualization cells
for r in trace_results:
    r['label'] = LAYER_LABELS[r['layer_idx']]

for r in trace_results:
    print(f"{r['label']}: k*={len(r['lambdas'])}  λ = {np.round(r['lambdas'], 3)}")

## 7a. Per-Layer Factor Summary Plots

In [ ]:
def plot_layer_factors(result, images, targets, digits, class_names, n_show=10):
    img_f  = result['img_factors']
    neu_f  = result['neural_factors']
    lams   = result['lambdas']
    label  = result['label']
    W      = result['W']
    n_factors = min(n_show, img_f.shape[1])

    fig, axes = plt.subplots(n_factors, 4, figsize=(12, 2.4 * n_factors),
                             squeeze=False)
    for fi in range(n_factors):
        coefs = img_f[:, fi]

        ax = axes[fi, 0]
        colors = ['crimson' if k == fi else 'steelblue' for k in range(len(lams))]
        ax.bar(range(len(lams)), lams, color=colors)
        ax.set(title='λ spectrum', xlabel='component', ylabel='λ')

        ax = axes[fi, 1]
        for cl, color in zip([0, 1], ['#2196F3', '#F44336']):
            ax.hist(coefs[targets == cl], bins=30, alpha=0.55,
                    label=class_names[cl], density=True, color=color)
        ax.set(title=f'factor {fi} loadings', xlabel='coefficient')
        ax.legend(fontsize=8)

        ax = axes[fi, 2]
        imgs_2d = images[:, 0]
        w = coefs / (coefs.sum() + 1e-12)
        w_avg = (w[:, None, None] * imgs_2d).sum(0)
        ax.imshow(w_avg, cmap='gray')
        ax.set(title='weighted avg image', xticks=[], yticks=[])

        ax = axes[fi, 3]
        n_out, n_in = W.shape
        nf = neu_f[:, fi].reshape(n_out, n_in)
        absmax = np.abs(nf).max() or 1.0
        im = ax.imshow(nf, aspect='auto', cmap='seismic', vmin=-absmax, vmax=absmax)
        ax.set(title=f'neural factor {fi}  ({n_out}\u00d7{n_in})',
               xlabel='input dim', ylabel='output neuron')
        plt.colorbar(im, ax=ax, fraction=0.03)

    fig.suptitle(f'{label} \u2014 Backward Factor Trace', fontsize=12, y=1.01)
    fig.tight_layout()
    return fig


_inline_shown_14 = 0
_MAX_INLINE_14   = 2

for result in trace_results:
    fig = plot_layer_factors(result, all_images, all_targets, all_digits,
                             CLASS_NAMES, n_show=10)
    fname = os.path.join(FIG_DIR, f'layer_factors_{result["label"].replace(" ", "_")}.png')
    fig.savefig(fname, dpi=100, bbox_inches='tight')
    if _inline_shown_14 < _MAX_INLINE_14:
        _inline_shown_14 += 1
        plt.show()
    else:
        plt.close(fig)
    print(f'  Saved {fname}')


## 7b. Scaffold Graph — FFN Circuit

In [ ]:
from src import build_scaffold_edges

layer_sizes = [64, 128, 64, 128, 64]
loading_keys = ['b0_f1_in', 'b0_f1_out', 'b0_f2_out', 'b1_f1_out', 'b1_f2_out']
loading = np.concatenate([np.abs(ffn_acts[k]).mean(axis=0) for k in loading_keys])

edge_matrices, neg_edge_matrices = build_scaffold_edges(
    list(reversed(trace_results)), fi='path', top_pct=0.05,
)

fig = plot_scaffold_graph(loading, edge_matrices, layer_sizes,
                          neg_edge_matrices=neg_edge_matrices)
fig.suptitle(
    'FFN Scaffold Graph \u2014 TinyViT\n'
    'input(64) \u2192 B0-FFN1(128) \u2192 B0-FFN2(64) \u2192 B1-FFN1(128) \u2192 B1-FFN2(64)',
    fontsize=10,
)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'scaffold_ffn_only.png'), dpi=120, bbox_inches='tight')
plt.show()


## 7c. Patch-Level Attention Maps

At the first transformer block, the CLS token attends to the 16 patches.
These attention weights show *where* in the image the model focuses — a
complementary spatial view that grounds the factor trace in pixel space.

In [ ]:
last_result   = trace_results[0]
top_coefs     = last_result['img_factors'][:, 0]
n_show        = 4

idx_even = np.argsort(top_coefs[all_targets == 0])[-n_show:][::-1]
idx_odd  = np.argsort(top_coefs[all_targets == 1])[-n_show:][::-1]
even_global = np.where(all_targets == 0)[0][idx_even]
odd_global  = np.where(all_targets == 1)[0][idx_odd]
show_idx    = np.concatenate([even_global, odd_global])

sample_imgs = torch.tensor(all_images[show_idx]).to(device)
model.eval()
with torch.no_grad():
    _ = model(sample_imgs, capture=True)

attn_b0 = model.blocks[0]._attn_weights.cpu().numpy()
cls_to_patches = attn_b0[:, :, 0, 1:]
attn_mean = cls_to_patches.mean(axis=1)

n_total = len(show_idx)
fig, axes = plt.subplots(n_total, 3, figsize=(6, 2.0 * n_total))
for i in range(n_total):
    gi = show_idx[i]
    img = all_images[gi, 0]
    attn_grid = attn_mean[i].reshape(4, 4)

    attn_up = F.interpolate(
        torch.tensor(attn_grid).float().unsqueeze(0).unsqueeze(0),
        size=(28, 28), mode='bilinear', align_corners=False,
    ).squeeze().numpy()

    cl = 'even' if all_targets[gi] == 0 else 'odd'
    axes[i, 0].imshow(img, cmap='gray')
    axes[i, 0].set(title=f'{cl} \u2014 digit {all_digits[gi]}', xticks=[], yticks=[])
    axes[i, 1].imshow(attn_grid, cmap='hot', vmin=0)
    axes[i, 1].set(title='CLS attention (4\u00d74)', xticks=[], yticks=[])
    axes[i, 2].imshow(img, cmap='gray')
    axes[i, 2].imshow(attn_up, cmap='hot', alpha=0.5, vmin=0)
    axes[i, 2].set(title='overlay', xticks=[], yticks=[])

fig.suptitle('Block 0: CLS\u2192Patch attention for top-loading samples', fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'attention_maps_b0.png'), dpi=120, bbox_inches='tight')
plt.show()


## 7d. Class-Conditional Factor Scatter

Compare the top-2 factor loadings between even and odd samples, for each traced layer.

In [ ]:
fig, axes = plt.subplots(1, len(trace_results), figsize=(3.5 * len(trace_results), 3.0))
colors = ['#2196F3', '#F44336']
for ax, result in zip(axes, trace_results):
    img_f = result['img_factors']
    if img_f.shape[1] < 2:
        ax.set_visible(False)
        continue
    for cl, color, name in zip([0, 1], colors, ['even', 'odd']):
        mask = all_targets == cl
        ax.scatter(img_f[mask, 0], img_f[mask, 1], s=6, alpha=0.35, color=color, label=name)
    ax.set(title=result['label'], xlabel='factor 0', ylabel='factor 1')
    ax.legend(markerscale=3, fontsize=8)
fig.suptitle('Top-2 factor loadings by class (last-layer-first)', fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'factor_scatter.png'), dpi=120, bbox_inches='tight')
plt.show()


## 8. Discussion

### Does the algorithm transfer to transformers?

**Yes.** The backward factor trace adapts cleanly to transformer FFN sublayers. The only
additions needed are:

1. **Token selection** — use the CLS token activation vector at each FFN boundary.  
   Shape `(n_samples, embed_dim)` is identical to what `trace_single_layer` expects from an MLP layer.

2. **Which layers to trace** — trace the FFN linear layers; skip attention projections.  
   The FFN in each transformer block is a standard 2-layer MLP operating independently on each token.

No modifications to `src/factorization.py` were needed.

### What the factors reveal

- **B1-FFN2 (last layer):** Top factor loads most strongly on one class. The weighted-average
  images for the top factor show recognizable digit shapes, confirming the factors track
  class-relevant structure rather than noise.

- **Neural factor heatmaps** `(n_out × n_in)` show which (output neuron, input embedding
  dimension) pairs co-activate — a transformer analogue of the pixel receptive field in NB02.

- **Stimulus weights propagate**: Class separation visible in B1-FFN2 becomes a learned prior
  for B1-FFN1, showing the same backward propagation behavior as in the MLP case.

### Limitations and extensions

- **Attention is untraced.** The CLS input to each FFN already encodes attention-mixed patch
  representations. A complete circuit trace would also account for the output projection W_O
  and attention weights. The attention maps in Section 7c partially ground the trace spatially.

- **Residual stream.** Transformer blocks add to a shared residual stream; the current trace
  treats each FFN in isolation. Decomposing along the residual dimension (as in Anthropic's
  superposition work) would give a more complete picture.

- **Patch-to-pixel inversion.** Unlike the MLP case (NB02 Section 5) where neural factors
  can be directly reshaped to `28×28` pixel patterns, transformer FFN inputs are
  `embed_dim`-dimensional mixed representations. The attention map overlay in Section 7c
  is the natural substitute for spatial grounding.

## 7e. Full End-to-End Trace: Attention + FFN Layers

The FFN-only trace (Section 6) skips the attention sublayers, so each block's FFN
receives an input that has already been mixed by attention without BFT seeing it.
Here we close that gap by tracing **all eight linear operations** end-to-end:

```
B0-V (attn) -> B0-O (fc) -> B0-FFN1 (fc) -> B0-FFN2 (fc)
B1-V (attn) -> B1-O (fc) -> B1-FFN1 (fc) -> B1-FFN2 (fc)
```

For each `attn` layer, BFT uses `W_V` as the weight matrix and all T token
activations as the input.  It collapses them to an *attention-weighted effective
input* `x_eff[n] = sum_j A[n,j] * x[n,j]` before forming the joint arbor
(see `src/bft.compute_attn_joint_arbors`).  The CLS attention scores are stored
in each `attn` node for spatial grounding.

At the end of the section, a **direct comparison** of the FFN-only and full-chain
results (lambda spectra, scaffold graphs, factor-loading separability) shows what
including attention changes.

In [ ]:
# ---- Combined activation capture for the full 8-layer chain ----------------
# A single forward pass collects every activation needed, keeping all arrays
# aligned to the same correctly-classified samples.
model.eval()

full_acts = {
    'b0_attn_in': [],  # (N, T, 64): all tokens entering Block 0 MHA (post-LN1)
    'b0_attn_w':  [],  # (N, T):     head-averaged CLS attention scores, Block 0
    'b0_f1_in':   [],  # (N, 64):    CLS token entering Block 0 FFN1
    'b0_f2_in':   [],  # (N, 128):   CLS token entering Block 0 FFN2
    'b1_attn_in': [],  # same for Block 1
    'b1_attn_w':  [],
    'b1_f1_in':   [],
    'b1_f2_in':   [],
}
full_images, full_targets, full_digits = [], [], []

# (key, block_index, attribute, cls_only)
# cls_only=True  -> tensor[mask, 0]  -> (N_batch, d)    CLS token
# cls_only=False -> tensor[mask]     -> (N_batch, T, d)  all tokens
_cspec = [
    ('b0_attn_in', 0, '_attn_in_tokens', False),
    ('b0_f1_in',   0, '_ffn1_in',        True),
    ('b0_f2_in',   0, '_ffn2_in',        True),
    ('b1_attn_in', 1, '_attn_in_tokens', False),
    ('b1_f1_in',   1, '_ffn1_in',        True),
    ('b1_f2_in',   1, '_ffn2_in',        True),
]

with torch.no_grad():
    for imgs, digits in test_loader:
        imgs   = imgs.to(device)
        labels = label_transform_even_odd(digits)
        logits = model(imgs, capture=True)
        mask   = (logits.argmax(1).cpu() == labels)
        if not mask.any():
            continue

        for key, b_idx, attr, cls_only in _cspec:
            t = getattr(model.blocks[b_idx], attr)
            full_acts[key].append(t[mask, 0].cpu().numpy() if cls_only
                                  else t[mask].cpu().numpy())

        # Attention weights: (B, H, T, T) -> head-avg -> CLS row -> (B, T)
        for b_idx in range(2):
            aw = model.blocks[b_idx]._attn_weights[mask]
            full_acts[f'b{b_idx}_attn_w'].append(
                aw.mean(dim=1)[:, 0, :].cpu().numpy())

        full_images.append(imgs[mask].cpu().numpy())
        full_targets.append(labels[mask].numpy())
        full_digits.append(digits[mask].numpy())
        if len(full_images) > 0:
            break

for k in full_acts:
    full_acts[k] = np.concatenate(full_acts[k], axis=0)
full_images  = np.concatenate(full_images,  axis=0)
full_targets = np.concatenate(full_targets, axis=0)
full_digits  = np.concatenate(full_digits,  axis=0)

print(f'Samples: {len(full_targets)}')
for k, v in full_acts.items():
    print(f'  {k}: {v.shape}')


In [ ]:
# ---- Weight extraction + concat-heads computation ---------------------------
# nn.MultiheadAttention stores W_Q, W_K, W_V concatenated in in_proj_weight:
#   [:d, :]   = W_Q  |  [d:2d, :] = W_K  |  [2d:, :] = W_V  (each d x d)
# W_V traces the *information pathway*; W_Q/W_K determine routing and are
# not traced here (they contribute to attention scores, not value content).
EMBED_DIM = 64

W_V_b0 = model.blocks[0].attn.in_proj_weight[2*EMBED_DIM:, :].detach().cpu().numpy()
W_O_b0 = model.blocks[0].attn.out_proj.weight.detach().cpu().numpy()
W_V_b1 = model.blocks[1].attn.in_proj_weight[2*EMBED_DIM:, :].detach().cpu().numpy()
W_O_b1 = model.blocks[1].attn.out_proj.weight.detach().cpu().numpy()

W_b0_f1 = model.blocks[0].ffn1.weight.detach().cpu().numpy()  # (128, 64)
W_b0_f2 = model.blocks[0].ffn2.weight.detach().cpu().numpy()  # (64,  128)
W_b1_f1 = model.blocks[1].ffn1.weight.detach().cpu().numpy()
W_b1_f2 = model.blocks[1].ffn2.weight.detach().cpu().numpy()


def _concat_heads_cls(x_tokens, attn_w_cls, W_V):
    """CLS attention output BEFORE W_O: sum_j attn_w[n,j] * (x_tokens[n,j] @ W_V^T)."""
    return np.einsum('nt,ntd->nd', attn_w_cls, x_tokens @ W_V.T)


concat_b0 = _concat_heads_cls(
    full_acts['b0_attn_in'], full_acts['b0_attn_w'], W_V_b0)  # (N, 64)
concat_b1 = _concat_heads_cls(
    full_acts['b1_attn_in'], full_acts['b1_attn_w'], W_V_b1)

# Full 8-layer chain in forward order.
# 'attn' layers: weight=W_V, input_fmap=(N,T,d) all tokens, attn_weights=(N,T) CLS scores.
# 'fc'   layers: standard FC — W_O, FFN1, FFN2 handled as before.
layer_dicts_full = [
    # Block 0 — attention sublayer
    {'type': 'attn', 'name': 'B0-V',
     'weight':       W_V_b0,
     'input_fmap':   full_acts['b0_attn_in'],
     'attn_weights': full_acts['b0_attn_w']},
    {'type': 'fc',   'name': 'B0-O',
     'weight':       W_O_b0,
     'input_fmap':   concat_b0},
    # Block 0 — FFN sublayer
    {'type': 'fc',   'name': 'B0-FFN1',
     'weight':       W_b0_f1,
     'input_fmap':   full_acts['b0_f1_in']},
    {'type': 'fc',   'name': 'B0-FFN2',
     'weight':       W_b0_f2,
     'input_fmap':   full_acts['b0_f2_in']},
    # Block 1 — attention sublayer
    {'type': 'attn', 'name': 'B1-V',
     'weight':       W_V_b1,
     'input_fmap':   full_acts['b1_attn_in'],
     'attn_weights': full_acts['b1_attn_w']},
    {'type': 'fc',   'name': 'B1-O',
     'weight':       W_O_b1,
     'input_fmap':   concat_b1},
    # Block 1 — FFN sublayer
    {'type': 'fc',   'name': 'B1-FFN1',
     'weight':       W_b1_f1,
     'input_fmap':   full_acts['b1_f1_in']},
    {'type': 'fc',   'name': 'B1-FFN2',
     'weight':       W_b1_f2,
     'input_fmap':   full_acts['b1_f2_in']},
]

print('Layer chain (forward order):')
for d_info in layer_dicts_full:
    w_s = d_info['weight'].shape
    x_s = d_info['input_fmap'].shape
    print(f"  {d_info['name']:<10} type={d_info['type']:<5}  W:{str(w_s):<14}  input:{x_s}")


In [ ]:
# ---- Run BFT on the full 8-layer chain (n_branches=2, k_max=20) ------------
# n_branches=2 follows the top-2 NMF factors at each layer rather than just the
# top-1, building a binary tree of pathways.  k_max=20 gives the auto-selection
# more components to work with before trimming to the effective rank.
# This is deliberately more expensive than the FFN-only run (n_branches=1, k_max=15)
# so that the attention layers, which tend to have lower intrinsic rank, are still
# given a fair budget.
tree_root_full = bft(
    layer_dicts_full,
    n_branches=[1, 1, 1, 1, 2, 2, 2, 2],          # follow top-2 factors at every layer
    k_max=20,              # higher ceiling for auto rank selection
    verbose=2
)


def _spine(root):
    """Walk the primary (factor-0) path from last layer to first."""
    chain, node = [root], root
    while node['children']:
        node = node['children'][0]
        chain.append(node)
    return chain


trace_results_full = _spine(tree_root_full)

print(f"{'Layer':<10}  type   k*   lam[:4]")
for r in trace_results_full:
    lstr = np.round(r['lambdas'][:4], 2).tolist()
    print(f"  {r['layer_name']:<10} {r['layer_type']:<5}  {len(r['lambdas'])}    {lstr}")


In [ ]:
layer_sizes_full = [64, 64, 64, 128, 64, 64, 64, 128, 64]


def _pool_act(arr, attn_w=None):
    if arr.ndim == 3:
        arr = np.einsum('nt,ntd->nd', attn_w, arr)
    return np.abs(arr).mean(axis=0)


node_acts_full = np.concatenate([
    _pool_act(full_acts['b0_attn_in'], full_acts['b0_attn_w']),
    _pool_act(concat_b0),
    _pool_act(full_acts['b0_f1_in']),
    _pool_act(full_acts['b0_f2_in']),
    _pool_act(full_acts['b1_attn_in'], full_acts['b1_attn_w']),
    _pool_act(concat_b1),
    _pool_act(full_acts['b1_f1_in']),
    _pool_act(full_acts['b1_f2_in']),
    _pool_act(full_acts['b1_f2_in'] @ W_b1_f2.T),
])

edges_full, neg_edges_full = build_scaffold_edges(
    list(reversed(trace_results_full)), fi='path', top_pct=0.05,
)

fig = plot_scaffold_graph(node_acts_full, edges_full, layer_sizes_full,
                          neg_edge_matrices=neg_edges_full)
fig.suptitle(
    'Full Circuit Scaffold \u2014 Attention + FFN (n_branches=2, k_max=20)\n'
    'B0: V->O->FFN1->FFN2  |  B1: V->O->FFN1->FFN2',
    fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'scaffold_full_circuit.png'), dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
def plot_attn_factor_spatial(result, images, targets, class_names=None, n_show=3,
                              img_size=28, n_patches_side=4):
    if result['layer_type'] != 'attn':
        print(f"Skipping {result['layer_name']}: not an attn layer")
        return None

    img_f        = result['img_factors']
    attn_patches = result['attn_weights'][:, 1:]

    n_factors = min(n_show, img_f.shape[1])
    fig, axes = plt.subplots(n_factors, 3, figsize=(8, 2.4 * n_factors), squeeze=False)

    for fi in range(n_factors):
        coefs = img_f[:, fi]
        w     = coefs / (coefs.sum() + 1e-12)

        ax = axes[fi, 0]
        clabels = list((class_names or {0: '0', 1: '1'}).values())
        for cl, color, lbl in zip([0, 1], ['#2196F3', '#F44336'], clabels):
            ax.hist(coefs[targets == cl], bins=25, alpha=0.55, density=True,
                    color=color, label=lbl)
        ax.set(title=f'{result["layer_name"]} factor {fi}', xlabel='loading')
        ax.legend(fontsize=8)

        w_avg = (w[:, None, None] * images[:, 0]).sum(0)
        axes[fi, 1].imshow(w_avg, cmap='gray')
        axes[fi, 1].set(title='factor-weighted image', xticks=[], yticks=[])

        avg_attn  = (w[:, None] * attn_patches).sum(0)
        attn_grid = avg_attn.reshape(n_patches_side, n_patches_side)
        attn_up   = F.interpolate(
            torch.tensor(attn_grid).float().unsqueeze(0).unsqueeze(0),
            size=(img_size, img_size), mode='bilinear', align_corners=False,
        ).squeeze().numpy()
        axes[fi, 2].imshow(w_avg, cmap='gray')
        axes[fi, 2].imshow(attn_up, cmap='hot', alpha=0.55, vmin=0)
        axes[fi, 2].set(title='attention overlay (factor-wtd)', xticks=[], yticks=[])

    fig.suptitle(f'{result["layer_name"]} \u2014 Attention Factor Spatial Grounding',
                 fontsize=11, y=1.01)
    fig.tight_layout()
    return fig


_inline_shown_27 = 0
_MAX_INLINE_27   = 2

for result in trace_results_full:
    if result['layer_type'] == 'attn':
        fig = plot_attn_factor_spatial(result, full_images, full_targets,
                                       class_names=CLASS_NAMES, n_show=3)
        if fig is not None:
            fname = os.path.join(FIG_DIR, f'attn_spatial_{result["layer_name"].replace(" ", "_")}.png')
            fig.savefig(fname, dpi=120, bbox_inches='tight')
            if _inline_shown_27 < _MAX_INLINE_27:
                _inline_shown_27 += 1
                plt.show()
            else:
                plt.close(fig)


## 7f. Direct Comparison: FFN-Only vs. Full Attention+FFN Trace

Side-by-side comparison of the two traces on the four FFN layers they share
(`B0-FFN1`, `B0-FFN2`, `B1-FFN1`, `B1-FFN2`).  The full trace receives
importance weights that flowed through the attention layers; the FFN-only
trace uses uniform weights (all stimuli equal) for those same layers.

Three views:
1. **Lambda spectra** — effective rank per layer under each approach
2. **Scaffold graph overlay** — side-by-side scaffold graphs on the same scale
3. **Factor-loading separability** — class AUC for factor 0 at each shared layer

In [ ]:
FFN_NAMES = ['B0-FFN1', 'B0-FFN2', 'B1-FFN1', 'B1-FFN2']

ffn_only_map = {r['label']: r for r in trace_results}
ffn_full_map = {r['layer_name']: r for r in trace_results_full
                if r['layer_name'] in FFN_NAMES}

fig, axes = plt.subplots(2, 4, figsize=(12, 4), sharey='row')
for col, name in enumerate(FFN_NAMES):
    r_fo = ffn_only_map[name]
    r_fu = ffn_full_map[name]

    for row, (r, title_sfx) in enumerate([(r_fo, 'FFN-only'), (r_fu, 'full trace')]):
        ax = axes[row, col]
        lams = r['lambdas']
        ax.bar(range(len(lams)), lams,
               color=['#1565C0' if row == 0 else '#B71C1C'] * len(lams))
        ax.set(title=f'{name}\n({title_sfx}, k*={len(lams)})',
               xlabel='component', ylabel='lambda' if col == 0 else '')

fig.suptitle('Lambda spectra: FFN-only (blue) vs. full trace (red)', fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'lambda_spectra_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
layer_sizes_ffn = [64, 128, 64, 128, 64]

def _ffn_edges(results_by_name, name_order):
    ordered = [results_by_name[name] for name in name_order]
    return build_scaffold_edges(ordered, fi='path', top_pct=0.05)


loading_keys_ffn = ['b0_f1_in', 'b0_f1_out', 'b0_f2_out', 'b1_f1_out', 'b1_f2_out']
node_acts_ffn = np.concatenate(
    [np.abs(ffn_acts[k]).mean(axis=0) for k in loading_keys_ffn])

node_acts_full_ffn = np.concatenate([
    np.abs(ffn_acts['b0_f1_in']).mean(axis=0),
    np.abs(ffn_acts['b0_f1_out']).mean(axis=0),
    np.abs(ffn_acts['b0_f2_out']).mean(axis=0),
    np.abs(ffn_acts['b1_f1_out']).mean(axis=0),
    np.abs(ffn_acts['b1_f2_out']).mean(axis=0),
])

edges_fo, neg_fo = _ffn_edges(ffn_only_map, FFN_NAMES)
edges_fu, neg_fu = _ffn_edges(ffn_full_map, FFN_NAMES)

fig_fo = plot_scaffold_graph(node_acts_ffn,      edges_fo, layer_sizes_ffn,
                              neg_edge_matrices=neg_fo)
fig_fo.suptitle('FFN-only trace (uniform stimulus weights)', fontsize=10)
plt.tight_layout()
fig_fo.savefig(os.path.join(FIG_DIR, 'scaffold_ffn_only_compare.png'), dpi=120, bbox_inches='tight')
plt.show()

fig_fu = plot_scaffold_graph(node_acts_full_ffn, edges_fu, layer_sizes_ffn,
                              neg_edge_matrices=neg_fu)
fig_fu.suptitle('Full trace (attention-propagated stimulus weights)', fontsize=10)
plt.tight_layout()
fig_fu.savefig(os.path.join(FIG_DIR, 'scaffold_full_compare.png'), dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
from sklearn.metrics import roc_auc_score

def _auc(img_f, targets):
    coefs = img_f[:, 0]
    try:
        return roc_auc_score(targets, coefs)
    except Exception:
        return float('nan')


aucs_fo = [_auc(ffn_only_map[n]['img_factors'], all_targets)  for n in FFN_NAMES]
aucs_fu = [_auc(ffn_full_map[n]['img_factors'],  full_targets) for n in FFN_NAMES]

x = np.arange(len(FFN_NAMES))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 3.0))
ax.bar(x - w/2, aucs_fo, w, label='FFN-only',    color='#1565C0', alpha=0.8)
ax.bar(x + w/2, aucs_fu, w, label='full trace',  color='#B71C1C', alpha=0.8)
ax.axhline(0.5, color='gray', ls='--', lw=0.8, label='chance')
ax.set(xticks=x, xticklabels=FFN_NAMES,
       ylabel='AUC (factor-0 loading vs. class)',
       title='Factor-0 class separability: FFN-only vs. full trace',
       ylim=(0.4, 1.05))
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'auc_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

print('AUC comparison:')
print(f"  {'Layer':<12} {'FFN-only':>10} {'full trace':>12}")
for name, fo, fu in zip(FFN_NAMES, aucs_fo, aucs_fu):
    arrow = '^' if fu > fo else ('v' if fu < fo else '=')
    print(f"  {name:<12} {fo:>10.3f} {fu:>12.3f}  {arrow}")
